# Visual Inference Demo: Vesuvius Ink Detection
**Project:** Vesuvius Challenge - Ink Masking  
**Institution:** DHLAB, EPFL

This interactive notebook demonstrates the end-to-end inference pipeline for extracting carbon ink masks from highly degraded papyrus scans. Rather than running the bulk-processing script (`lib/inference.py`), this notebook breaks down the methodology step-by-step on a single sample fragment.

**Pipeline Stages:**
1. Macroscopic Pre-processing & CLAHE Enhancement
2. Foundation Model (NVlabs/RADIO) Feature Extraction
3. Sliding Window Inference (Probability Heatmap)
4. Hybrid Post-Processing & Morphological Reconnection

In [ ]:
import os
import cv2
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
from PIL import Image

# Import architecture from our local library
import sys
sys.path.append('../lib')
from inference import RadioSegmentationHead, apply_hybrid_post_processing

# --- CONFIGURATION ---
# Path to the sample images hosted on the GitHub repository
SAMPLE_IMG_PATH = "../data/sample_images/229.tif"
WEIGHTS_PATH = "../weights/radio_production_model.pth"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[*] Using computation device: {device.upper()}")

## 1. Macroscopic Pre-processing & Photometric Enhancement
Historical papyri suffer from severe illumination variances, carbonized black spots, and faded ink. To standardize the input for the Vision Transformer, we apply **Contrast Limited Adaptive Histogram Equalization (CLAHE)**.

To prevent color distortion, CLAHE is strictly applied to the Lightness (L) channel of the LAB color space before being merged back into RGB.

In [ ]:
def apply_rgb_clahe(img_rgb, clip_limit=2.0, tile_grid=(8, 8)):
    """Applies CLAHE strictly to the luminance channel to preserve original colors."""
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

# Load the raw sample image
raw_img = np.array(Image.open(SAMPLE_IMG_PATH).convert('RGB'))

# Apply Enhancement
clahe_img = apply_rgb_clahe(raw_img)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(raw_img)
axes[0].set_title("Original Papyrus Scan", fontsize=14)
axes[0].axis('off')

axes[1].imshow(clahe_img)
axes[1].set_title("CLAHE Enhanced (Input to Model)", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 2. Model Initialization
We load the frozen Vision Foundation Model (**NVlabs/RADIO v2.5-l**) to extract rich spatial embeddings. These embeddings are then passed to our custom trained Multi-Layer Perceptron (MLP) segmentation head.

In [ ]:
print("[*] Loading NVlabs/RADIO Foundation Model...")
radio_v2 = torch.hub.load('NVlabs/RADIO', 'radio_model', version='radio_v2.5-l', skip_validation=True).to(device)
radio_v2.eval()

print("[*] Loading Custom MLP Segmentation Head...")
head = RadioSegmentationHead(input_dim=4096).to(device)

if os.path.exists(WEIGHTS_PATH):
    state_dict = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
    clean_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    head.load_state_dict(clean_state_dict)
    print("[+] Model weights loaded successfully.")
else:
    print(f"[WARNING] Weights not found at {WEIGHTS_PATH}. Please download them from the release page.")
    
head.eval()

# Standard ImageNet normalization required by the Foundation Model
transform = T.Compose([
    T.Resize((2048, 2048), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 3. Sliding Window Inference
Because the macroscopic papyrus scans exceed the memory capacity of standard GPUs, we process the image using a **sliding window technique** with a 50% spatial overlap ($stride = 128$ pixels). 

The overlapping predictions are averaged to eliminate boundary grid artifacts, resulting in a smooth, continuous probability heatmap.

In [ ]:
patch_size = 256
stride = patch_size // 2
original_h, original_w = raw_img.shape[:2]

# Divisible Padding
pad_h = math.ceil(original_h / patch_size) * patch_size - original_h
pad_w = math.ceil(original_w / patch_size) * patch_size - original_w
img_divisible = cv2.copyMakeBorder(raw_img, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=[0, 0, 0])
padded_h, padded_w = img_divisible.shape[:2]

full_prob_grid = np.zeros((padded_h, padded_w), dtype=np.float32)
count_grid = np.zeros((padded_h, padded_w), dtype=np.float32)

print("[*] Starting Sliding Window Inference...")

# Sequential inference for demonstration clarity
for y in range(0, padded_h - patch_size + 1, stride):
    for x in range(0, padded_w - patch_size + 1, stride):
        patch_img = img_divisible[y:y+patch_size, x:x+patch_size]
        input_tensor = transform(Image.fromarray(patch_img)).unsqueeze(0).to(device)
        
        with torch.no_grad():
            with torch.amp.autocast(device, dtype=torch.float16):
                summary, spatial = radio_v2(input_tensor)
                if len(spatial.shape) == 4:
                    spatial = spatial.permute(0, 2, 3, 1).reshape(spatial.shape[0], -1, spatial.shape[1])

                num_tokens = spatial.shape[1]
                summary_expanded = summary.unsqueeze(1).expand(-1, num_tokens, -1)
                combined_tokens = torch.cat([spatial, summary_expanded], dim=-1).view(-1, 4096)

                logits = head(combined_tokens)
                grid_size = int(math.sqrt(num_tokens))
                probs = torch.sigmoid(logits).view(grid_size, grid_size).cpu().numpy().astype(np.float32)

        # Upscale feature map to patch resolution and accumulate
        probs_resized = cv2.resize(probs, (patch_size, patch_size), interpolation=cv2.INTER_CUBIC)
        full_prob_grid[y:y+patch_size, x:x+patch_size] += probs_resized
        count_grid[y:y+patch_size, x:x+patch_size] += 1.0

# Average the overlapping predictions
full_prob_grid = full_prob_grid / np.maximum(count_grid, 1.0)
full_prob_grid = full_prob_grid[:original_h, :original_w]

# Visualization
plt.figure(figsize=(10, 10))
plt.imshow(full_prob_grid, cmap='inferno', vmin=0, vmax=1)
plt.title("AI Probability Heatmap (Logits)", fontsize=16)
plt.axis('off')
plt.colorbar(fraction=0.046, pad=0.04)
plt.show()

## 4. Hybrid Post-Processing & Morphological Reconnection
To transform the continuous probability heatmap into a crisp binary ground truth mask, we apply our targeted post-processing sequence:

1. **Edge Artifact Suppression:** A 7x7 dilation is used to zero-out hallucinated ink on physical tears.
2. **Semantic-Guided Contours:** A logical `AND` fuses high-confidence AI stencils ($P > 0.55$) with an adaptive structural contrast threshold ($C=4$).
3. **Morphological Reconnection:** A localized 4x4 closing operation acts as structural glue, reconnecting disjointed strokes of faded Greek characters.

In [ ]:
# 1. Fast Background Gap Detection
gray = cv2.cvtColor(raw_img, cv2.COLOR_RGB2GRAY)
blurred = cv2.GaussianBlur(gray, (11, 11), 0)
_, white_bg = cv2.threshold(blurred, 200, 255, cv2.THRESH_BINARY)
raw_silhouette = cv2.bitwise_not(white_bg)
contours, _ = cv2.findContours(raw_silhouette, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

solid_papyrus = np.zeros_like(gray)
for contour in contours:
    if cv2.contourArea(contour) > 500:
        cv2.drawContours(solid_papyrus, [contour], -1, 255, thickness=cv2.FILLED)

gap_mask_full = cv2.bitwise_or(cv2.bitwise_not(solid_papyrus), white_bg) == 255

# 2. Execute Hybrid Post-Processing from our library
final_ink_mask = apply_hybrid_post_processing(raw_img, full_prob_grid, gap_mask_full)

# 3. Generate Final Output Visualizations
final_mask_display = ((1 - final_ink_mask) * 255).astype(np.uint8) # Inverted for visibility

overlay_img = raw_img.copy()
alpha = 0.4
ink_pixels = (final_ink_mask == 1)
overlay_img[ink_pixels] = (overlay_img[ink_pixels] * (1 - alpha) + np.array([255, 0, 0]) * alpha).astype(np.uint8)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

axes[0].imshow(final_mask_display, cmap='gray')
axes[0].set_title("Final Binarized Ground Truth Mask", fontsize=16)
axes[0].axis('off')

axes[1].imshow(overlay_img)
axes[1].set_title("Quality Control Overlay (Red = Detected Ink)", fontsize=16)
axes[1].axis('off')

plt.tight_layout()
plt.show()